# PyVRP Solver

Runs PyVRP on all benchmark instances (no time limit, runs until convergence). Output uploaded as `elfateh/dana-results-pyvrp`.

In [ ]:
import os, subprocess, sys, json, time, csv, math

subprocess.run(["pip", "install", "-q", "pyvrp", "numpy", "kagglehub", "requests"], check=True)

INSTANCE_DIR = "/kaggle/working/instances"
if not os.path.exists(INSTANCE_DIR):
    try:
        import kagglehub
        path = kagglehub.dataset_download("elfateh/dana-instances")
        subprocess.run(["cp", "-r", os.path.join(path, "*"), "/kaggle/working/instances/"], check=True)
    except Exception as e:
        print(f"Kaggle download failed, trying GitHub: {e}")
        import requests as _requests
        BASE_URL = "https://raw.githubusercontent.com/PyVRP/Instances/main"
        SETS = {
            "cordeau": ("MDVRPTW", [f"PR{i}{s}.vrp" for i in range(11, 25) for s in ("A", "B")]),
            "solomon": ("VRPTW/Solomon", ["C101.vrp","C102.vrp","C103.vrp","C104.vrp","C105.vrp","C106.vrp","C107.vrp","C108.vrp","C109.vrp","C201.vrp","C202.vrp","C203.vrp","C204.vrp","C205.vrp","C206.vrp","C207.vrp","C208.vrp","R101.vrp","R102.vrp","R103.vrp","R104.vrp","R105.vrp","R106.vrp","R107.vrp","R108.vrp","R109.vrp","R110.vrp","R111.vrp","R112.vrp","R201.vrp","R202.vrp","R203.vrp","R204.vrp","R205.vrp","R206.vrp","R207.vrp","R208.vrp","R209.vrp","R210.vrp","R211.vrp","RC101.vrp","RC102.vrp","RC103.vrp","RC104.vrp","RC105.vrp","RC106.vrp","RC107.vrp","RC108.vrp","RC201.vrp","RC202.vrp","RC203.vrp","RC204.vrp","RC205.vrp","RC206.vrp","RC207.vrp","RC208.vrp"]),
            "gehring": ("VRPTW", [f"{t}{n}_10_{i}.vrp" for t in ("C","R","RC") for n in (1,2) for i in range(1,11)]),
            "x_instances": ("CVRP", ["X-n101-k25.vrp","X-n106-k14.vrp","X-n110-k13.vrp","X-n115-k10.vrp","X-n120-k6.vrp","X-n125-k30.vrp","X-n129-k18.vrp","X-n134-k13.vrp","X-n139-k10.vrp","X-n143-k7.vrp","X-n153-k22.vrp","X-n157-k13.vrp","X-n162-k11.vrp","X-n167-k10.vrp","X-n176-k26.vrp","X-n181-k23.vrp","X-n186-k15.vrp","X-n190-k8.vrp","X-n195-k51.vrp","X-n200-k36.vrp","X-n204-k19.vrp","X-n209-k16.vrp","X-n214-k11.vrp","X-n219-k73.vrp","X-n223-k34.vrp","X-n228-k23.vrp","X-n233-k16.vrp","X-n237-k14.vrp","X-n242-k48.vrp","X-n247-k50.vrp"]),
        }
        for sname, (rdir, files) in SETS.items():
            dest = os.path.join(INSTANCE_DIR, sname)
            os.makedirs(dest, exist_ok=True)
            for fname in files:
                url = f"{BASE_URL}/{rdir}/{fname}"
                path_d = os.path.join(dest, fname)
                if not os.path.exists(path_d):
                    try:
                        r = _requests.get(url, timeout=30)
                        if r.status_code == 200:
                            with open(path_d, "w") as f:
                                f.write(r.text)
                    except Exception:
                        pass
        print("Downloaded instances from GitHub")

from pyvrp import read, Model
from pyvrp.stop import NoImprovement


In [ ]:
# No time limit - runs until convergence
BENCHMARKS = {
    "cordeau": {"dir": "cordeau", "problem_type": "mdvrptw"},
    "solomon": {"dir": "solomon", "problem_type": "vrptw"},
    "gehring": {"dir": "gehring", "problem_type": "vrptw"},
    "x_instances": {"dir": "x_instances", "problem_type": "cvrp"},
}
OUT_CSV = "/kaggle/working/pyvrp_results.csv"
rows = []

for bench_name, bench_info in BENCHMARKS.items():
    bench_dir = os.path.join(INSTANCE_DIR, bench_info["dir"])
    if not os.path.exists(bench_dir):
        print(f"SKIP: {bench_dir} not found")
        continue
    instance_files = sorted([f for f in os.listdir(bench_dir) if f.endswith(".vrp")])
    print(f"\n=== {bench_name} ({len(instance_files)} instances) ===")

    for fname in instance_files:
        inst_file = os.path.join(bench_dir, fname)
        inst_name = fname.replace(".vrp", "")
        t0 = time.time()
        try:
            data = read(inst_file)
            model = Model.from_data(data)
            result_obj = model.solve(stop=NoImprovement(500), seed=42, display=False)
            elapsed = time.time() - t0

            best = result_obj.best
            num_depots = data.num_depots
            depots = data.depots()
            clients = data.clients()
            loc_coords = [(d.x, d.y) for d in depots] + [(c.x, c.y) for c in clients]

            def loc_dist(i, j):
                dx = loc_coords[i][0] - loc_coords[j][0]
                dy = loc_coords[i][1] - loc_coords[j][1]
                return math.sqrt(dx * dx + dy * dy)

            routes = []
            raw_cost = 0.0
            for route in best.routes():
                visits = list(route.visits())
                if not visits:
                    continue
                sd, ed = int(route.start_depot()), int(route.end_depot())
                raw_cost += loc_dist(sd, visits[0])
                for i in range(len(visits) - 1):
                    raw_cost += loc_dist(visits[i], visits[i + 1])
                raw_cost += loc_dist(visits[-1], ed)
                routes.append(visits)

            rows.append({
                "instance": inst_name,
                "solver": "pyvrp",
                "benchmark": bench_name,
                "problem_type": bench_info["problem_type"],
                "cost": raw_cost,
                "raw_euclidean_cost": raw_cost,
                "cost_unit": "euclidean",
                "time_s": round(elapsed, 2),
                "status": "success",
                "error_msg": "",
            })
            print(f"  {inst_name}: {raw_cost:.2f} (pyvrp={result_obj.cost()}) ({elapsed:.1f}s)")
        except Exception as e:
            elapsed = time.time() - t0
            rows.append({
                "instance": inst_name,
                "solver": "pyvrp",
                "benchmark": bench_name,
                "problem_type": bench_info["problem_type"],
                "cost": None,
                "raw_euclidean_cost": None,
                "cost_unit": "euclidean",
                "time_s": round(elapsed, 2),
                "status": "error",
                "error_msg": str(e),
            })
            print(f"  {inst_name}: ERROR {e}")

with open(OUT_CSV, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["instance", "solver", "benchmark", "problem_type",
                                            "cost", "raw_euclidean_cost", "cost_unit", "time_s",
                                            "status", "error_msg"])
    writer.writeheader()
    writer.writerows(rows)

print(f"\nSaved {len(rows)} results to {OUT_CSV}")


In [ ]:
DATASET_DIR = "/kaggle/working/dana-results-pyvrp"
os.makedirs(DATASET_DIR, exist_ok=True)
subprocess.run(["cp", OUT_CSV, os.path.join(DATASET_DIR, "pyvrp_results.csv")], check=True)

with open(os.path.join(DATASET_DIR, "dataset-metadata.json"), "w") as f:
    json.dump({
        "title": "PyVRP Results",
        "id": "elfateh/dana-results-pyvrp",
        "licenses": [{"name": "CC0-1.0"}],
    }, f)

# Always try version first (dataset already exists), fall back to create
r = subprocess.run(
    ["kaggle", "datasets", "version", "-p", DATASET_DIR, "-m", "Updated results", "--dir-mode", "zip"],
    capture_output=True, text=True,
)
if r.returncode != 0:
    r2 = subprocess.run(
        ["kaggle", "datasets", "create", "-p", DATASET_DIR, "--dir-mode", "zip"],
        capture_output=True, text=True,
    )
    if r2.returncode != 0:
        print(f"Dataset upload failed: {r2.stderr.strip()}")
    else:
        print("Dataset created successfully.")
else:
    print("Dataset version updated successfully.")